# 01 - Data Ingestion

**Purpose:** Download, validate, and align time series data from two sources:
1. London Datastore Economic Fairness indicators (shock signals)
2. Food Foundation Food Insecurity Tracker (response signal)

**Output:** `/data/processed/aligned_timeseries.parquet`

In [ ]:
import polars as pl
import pandas as pd
import requests
from pathlib import Path
from datetime import datetime

# Project paths
PROJECT_ROOT = Path.cwd().parent
RAW_DATA = PROJECT_ROOT / "data" / "raw"
PROCESSED_DATA = PROJECT_ROOT / "data" / "processed"

print(f"Project root: {PROJECT_ROOT}")
print(f"Raw data: {RAW_DATA}")
print(f"Processed data: {PROCESSED_DATA}")

## 1. Download Economic Fairness Data

Source: London Datastore - State of London Economic Fairness Indicator Matrix (November 2020)

In [ ]:
ECONOMIC_FAIRNESS_URL = "https://cdn-wp.datapress.cloud/london/20250618131339/Indicator-matrix-data-sheets-Nov-2020.xlsx"
ECONOMIC_FAIRNESS_FILE = RAW_DATA / "economic_fairness" / "indicator_matrix_nov_2020.xlsx"

def download_file(url: str, dest: Path, force: bool = False) -> Path:
    """
    Download a file from URL to destination path.
    
    Uses streaming to handle large files efficiently.
    Skips download if file exists unless force=True.
    
    Args:
        url: Source URL
        dest: Destination file path
        force: Re-download even if file exists
    
    Returns:
        Path to downloaded file
    """
    if dest.exists() and not force:
        print(f"File already exists: {dest}")
        return dest
    
    print(f"Downloading: {url}")
    dest.parent.mkdir(parents=True, exist_ok=True)
    
    response = requests.get(url, stream=True)
    response.raise_for_status()
    
    total_size = int(response.headers.get('content-length', 0))
    
    with open(dest, 'wb') as f:
        downloaded = 0
        for chunk in response.iter_content(chunk_size=8192):
            f.write(chunk)
            downloaded += len(chunk)
            if total_size:
                pct = (downloaded / total_size) * 100
                print(f"\rProgress: {pct:.1f}%", end="")
    
    print(f"\nSaved to: {dest}")
    return dest

In [ ]:
# Download Economic Fairness data
download_file(ECONOMIC_FAIRNESS_URL, ECONOMIC_FAIRNESS_FILE)

## 2. Explore Economic Fairness Excel Structure

The Excel file contains 42 sheets covering various economic indicators.
Key sheets for our analysis:
- **Fuel poverty** - Direct economic stress indicator (2003-2018)
- **Poverty / Absolute poverty** - Vulnerability measures (1994-present)
- **Disposable Income** - Spending capacity (2009-2018)
- **Employment gaps** - Job market stress (2004-present)

In [ ]:
import openpyxl

# Load workbook to inspect sheet names
wb = openpyxl.load_workbook(ECONOMIC_FAIRNESS_FILE, read_only=True)
print("Available sheets:")
for i, sheet_name in enumerate(wb.sheetnames, 1):
    print(f"  {i:2}. {sheet_name}")
wb.close()

### 2.1 Extract Fuel Poverty Time Series

Fuel poverty is a strong leading indicator - households struggling with energy costs often face food insecurity next.

In [ ]:
# Read Fuel Poverty sheet
fuel_poverty_raw = pd.read_excel(ECONOMIC_FAIRNESS_FILE, sheet_name='Fuel poverty', header=None)

# The data starts at row 4 (0-indexed: row 3), columns 0-2 contain Year, London %, England %
# Extract the clean time series
fuel_poverty_df = fuel_poverty_raw.iloc[4:20, 0:3].copy()
fuel_poverty_df.columns = ['year', 'london_pct', 'england_pct']
fuel_poverty_df = fuel_poverty_df.dropna()
fuel_poverty_df['year'] = fuel_poverty_df['year'].astype(int)

print("Fuel Poverty (% of households):")
print(fuel_poverty_df.to_string(index=False))

### 2.2 Extract Disposable Income Time Series

In [ ]:
# Read Disposable Income sheet
income_raw = pd.read_excel(ECONOMIC_FAIRNESS_FILE, sheet_name='Disposable Income', header=None)

# Data starts at row 3, columns 0-3 contain Period, London, UK, Rest of UK
income_df = income_raw.iloc[3:, 0:4].copy()
income_df.columns = ['period', 'london', 'uk', 'rest_of_uk']
income_df = income_df.dropna()

print("Median Equivalised Weekly Disposable Income (AHC, 2016 prices):")
print(income_df.to_string(index=False))

### 2.3 Extract Absolute Poverty Time Series (London)

In [ ]:
# Read Absolute Poverty sheet
poverty_raw = pd.read_excel(ECONOMIC_FAIRNESS_FILE, sheet_name='Absolute poverty', header=None)

# Structure: Row 7+ has data, columns vary by demographic group
# Column 0: Period, Column 2: London % (All people, BHC), Column 5: London % (All people, AHC)
poverty_df = poverty_raw.iloc[7:, [0, 2, 5]].copy()
poverty_df.columns = ['period', 'london_bhc_pct', 'london_ahc_pct']
poverty_df = poverty_df.dropna()

print("Absolute Poverty Rate - London (% below 60% 2010/11 median):")
print("BHC = Before Housing Costs, AHC = After Housing Costs")
print(poverty_df.head(15).to_string(index=False))

## 3. Load Food Foundation Data

Food insecurity survey data compiled from Food Foundation reports and FSA Food and You 2 survey.

**Sources:**
- [Food Foundation Tracker](https://foodfoundation.org.uk/initiatives/food-insecurity-tracking)
- [UK Food Security Report 2024](https://www.gov.uk/government/statistics/united-kingdom-food-security-report-2024)

In [ ]:
FOOD_INSECURITY_FILE = RAW_DATA / "food_insecurity" / "survey_rounds.csv"

food_insecurity_df = pl.read_csv(FOOD_INSECURITY_FILE)
print("Food Insecurity Survey Data (% of UK households):")
print(food_insecurity_df)

## 4. Data Quality Assessment

Before proceeding to analysis, let's assess data coverage and alignment.

In [ ]:
print("DATA COVERAGE SUMMARY")
print("=" * 60)
print("\n1. ECONOMIC FAIRNESS INDICATORS (Shock Signals)")
print(f"   Fuel Poverty:      {fuel_poverty_df['year'].min()}-{fuel_poverty_df['year'].max()} ({len(fuel_poverty_df)} years)")
print(f"   Disposable Income: {len(income_df)} periods (3-year rolling)")
print(f"   Absolute Poverty:  {len(poverty_df)} periods")

print("\n2. FOOD INSECURITY (Response Signal)")
print(f"   Survey rounds:     {len(food_insecurity_df)} data points")
print(f"   Date range:        {food_insecurity_df['survey_date'].min()} to {food_insecurity_df['survey_date'].max()}")

print("\n3. OVERLAP PERIOD")
print("   Economic data ends ~2018-2020")
print("   Food insecurity data starts March 2020")
print("   *** LIMITED OVERLAP - will need supplementary data ***")

## 5. Save Processed Data

Export cleaned time series for use in subsequent notebooks.

In [ ]:
PROCESSED_DATA.mkdir(parents=True, exist_ok=True)

# Convert pandas DataFrames to polars and save
pl.from_pandas(fuel_poverty_df).write_parquet(PROCESSED_DATA / "fuel_poverty.parquet")
pl.from_pandas(income_df).write_parquet(PROCESSED_DATA / "disposable_income.parquet")
pl.from_pandas(poverty_df).write_parquet(PROCESSED_DATA / "absolute_poverty.parquet")
food_insecurity_df.write_parquet(PROCESSED_DATA / "food_insecurity.parquet")

print("Saved processed data:")
for f in PROCESSED_DATA.glob("*.parquet"):
    print(f"  - {f.name}")

## 6. Next Steps

**Key Finding:** The Economic Fairness data (ending ~2018-2020) has limited overlap with Food Foundation data (starting March 2020). 

**Options:**
1. Use pre-pandemic economic indicators as baseline context
2. Seek updated Economic Fairness data from London Datastore
3. Supplement with more recent ONS data (claimant count, energy prices)

**Proceed to:** `02_exploratory_viz.ipynb` to visualise what we have and identify patterns.